 Copyright © Sorbonne University.

 This source code is licensed under the MIT license found in the
 LICENSE file in the root directory of this source tree.

# PLDAC - RL TABULAIRE


In this notebook, we will code a naive actor-critic algorithm in the tabular case using q-tables instead of the value function v.


## Install libraries

Installs the necessary Python and system libraries

In [1]:
try:
    from easypip import easyimport
except ModuleNotFoundError:
    from subprocess import run
    assert run(["pip", "install", "easypip"]).returncode == 0, "Could not install easypip"
    from easypip import easyimport

easyimport("swig")
easyimport("bbrl_utils").setup(maze_mdp=True)

import os
from pathlib import Path
from typing import List, Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
from bbrl_gymnasium.envs.maze_mdp import MazeMDPEnv
from bbrl_utils.notebook import tqdm
from mazemdp.mdp import Mdp
from mazemdp.toolbox import egreedy, egreedy_loc, sample_categorical, softmax
from mazemdp import random_policy
import bbrl_gymnasium  # noqa: F401

[easypip] Installing swig
[easypip] Installing bbrl_utils


Matplotlib backend: module://matplotlib_inline.backend_inline


/usr/local/lib/python3.11/dist-packages/bbrl_utils/notebook.py:46: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm  # noqa: F401
  if event.key is 'enter':



In [2]:
try:
    from easypip import easyimport
except ModuleNotFoundError:
    from subprocess import run
    assert run(["pip", "install", "easypip"]).returncode == 0, "Could not install easypip"
    from easypip import easyimport

import os
from pathlib import Path
from typing import List, Tuple

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np

from bbrl_gymnasium.envs.maze_mdp import MazeMDPEnv
from mazemdp.mdp import Mdp
from mazemdp.toolbox import egreedy, egreedy_loc, sample_categorical, softmax
from mazemdp import random_policy
import bbrl_gymnasium  # noqa: F401

# matplotlib.use("TkAgg")

from functools import partial
!pip install optuna
import optuna
import yaml
import hydra
from omegaconf import OmegaConf, DictConfig

import seaborn as sns
sns.set_theme()
import logging
optuna.logging.set_verbosity(optuna.logging.WARNING)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.6/383.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.8/231.8 kB 10.5 MB/s eta 0:00:00


# Step 1: Updating the naive Actor-critic algorithm with Q-values

We consider the naive actor-critic algorithm with a categorical policy.  
The algorithm learns a critic with the standard temporal difference mechanism  
using a learning rate \(\alpha_{critic}\).

We consider a **Q-value-based critic** \(Q(s, a)\) instead of a value-based critic \(V(s)\), which allows the algorithm to estimate the value of taking action \(a\) from state \(s\).  


---

## Critic Update Rules

We implement four different update rules for the Q-values:

1. **SARSA Update Rule**  
   $Q(s_t, a_t) ← Q(s_t, a_t) + alpha-critic [r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)]$

   This rule follows the **on-policy** approach, where the next action is drawn according to the policy.  

2. **Q-learning Update Rule**  
   $Q(s_t, a_t) ← Q(s_t, a_t) + alpha-critic [r_{t+1} + \gamma max_a Q(s_{t+1}, a) - max_a Q(s_t, a)]$
   
   This rule follows the **off-policy** approach, where the best possible action is chosen greedily.  

3. **"Mean" SARSA Update Rule**  
   $Q(s_t, a_t) ← Q(s_t, a_t) + alpha-critic [r_{t+1} + \gamma max_a Q(s_{t+1}, a) - \Sigma_a \pi(a|s_t)  Q(s_t, a)]$
   
   This rule leverages the **expected value** of the Q-function over all actions, weighted by the current policy.  

4. **Fourth Update Rule**  
   $Q(s_t, a_t) ← Q(s_t, a_t) + alpha-critic [r_{t+1} + \gamma max_a Q(s_{t+1}, a) - Q(s_t, a_t)]$  
   This rule uses a one-step update towards the maximum Q-value of the next state.  
   
   

---

## Actor Update

To update the actor, the general idea is the same, using the temporal difference error with another learning rate $\alpha_{actor}$.

However, naively applying the same learning rule would not ensure that the probabilities of all actions in a state sum to 1.
Besides, when the temporal difference error $\delta_t$ is negative, it may happen that the probability of an action gets negative or null, which raises an issue when applying renormalization.

So, instead of applying the naive rule, we apply the following one:
$$
\pi_{temp}(a_t|s_t) =  \begin{cases}
\pi^{(i)}(a_t|s_t) + \alpha_{actor} \delta_t & \mathrm{if } \pi^{(i)}(a_t|s_t) + \alpha_{actor} \delta_t > 10^{-8}\\
10^{-8} & \mathrm{otherwise.} \\
\end{cases}
$$

Then we can apply renormalization so that the probabilities of actions still sum to 1, with
$$
\forall a, \pi^{(i+1)}(a|s_t) = \frac{\pi_{temp}^{(i+1)}(a|s_t)} {\sum_{a'} \pi_{temp}^{(i+1)}(a'|s_t)}
$$ with
$$
\pi_{temp}^{(i+1)}(a|s_t) =  \begin{cases}
\pi_{temp}(a|s_t) & \mathrm{if } a = a_t\\
\pi^{(i)}(a|s_t) & \mathrm{otherwise.} \\
\end{cases}
$$

---

## Implementation:

We provide a Python implementation of the naive actor-critic algorithm with Q-values using the four update rules.  


---


In [9]:
def renormalize(
    mdp: MazeMDPEnv,
    policy: np.array,
    state: np.array,
) -> None:
    """
    Renormalize the probability of actions so that the sum of probabilities over actions is always 1.
    We made sure in the calling function that the probabilities of action never get negative when they are decreased.
    :param mdp: the mdp the agent is working in (to get the number of actions)
    :param policy: the current policy, before normalization
    :param state: the state where the actions need to be renormalized
    :return: nothing
    """

    policy[state] = policy[state] / np.sum(policy[state])


In [10]:
def perform_episode(
    mdp: MazeMDPEnv,
    policy: np.array,
    q: np.array,
    alpha_critic: float,
    alpha_actor: float,
    update_type: int,
    render: bool = True,
) -> int:
    """
    Perform an episode on the given mdp with the given policy using Q-values.
    :param update_type: int indicating the chosen update rule (1, 2, 3, 4).
    :return: the number of steps before it stops
    """
    state, _ = mdp.reset(uniform=True) # reninit env, get initial state -------------- s_t
    terminated = False
    truncated = False
    steps = 0

    while not (terminated or truncated):
        action = sample_categorical(policy[state])  #draw action from the policy-------------- a_t
        next_state, reward, terminated, truncated, _ = mdp.step(action)

        # max Q-value for the next state
        max_q_next = np.max(q[next_state]) # best q for next step -------------- s_{t+1}
        # next action according to the current policy (for SARSA)
        next_action = sample_categorical(policy[next_state]) # -------------- a_{t+1}

        # TD error (delta) based on the selected update type
        if update_type == 1:
            # 1: SARSA update rule
            delta = reward + mdp.gamma * q[next_state, next_action] * (1 - terminated) - q[state, action]

        elif update_type == 2:
            # 2: Q-learning update rule
            delta = reward + mdp.gamma * max_q_next * (1 - terminated) - np.max(q[state])

        elif update_type == 3:
            # 3: MEAN SARSA update rule
            sum = np.sum(policy[state] * q[state])
            delta = reward + mdp.gamma * max_q_next * (1 - terminated) - sum

        elif update_type == 4:
            # 4: Fourth update rule
            delta = reward + mdp.gamma * max_q_next * (1 - terminated) - q[state, action]
        else:
            raise ValueError("Invalid update type")

        # Update the Q-value with the delta
        q[state, action] += alpha_critic * delta

        # Update the policy with the new Q-values using the softmax function
        policy_temp = policy[state, action] + alpha_actor * delta
        policy[state, action] = max(policy_temp, 1e-8)
        renormalize(mdp, policy, state)

        # Move to the next state
        state = next_state
        steps += 1
        value_norm = np.linalg.norm(q)

    return value_norm, steps

Here comes the main actor-critic function. It should take the shown parameters as input and output a list of value norms and a list of number of steps,
corresponding to the evolution of these quantities through learning

In [11]:
def actor_critic_q(
    mdp: MazeMDPEnv,
    nb_episodes: int,
    alpha_critic: float,
    alpha_actor: float,
    update_type: int,
    render: bool = True,
) -> np.array:
    """
    Perform actor-critic training over a number of episodes with Q-values.
    :param update_type: int indicating the chosen update rule (1, 2, 3, 4).
    :return: a list of norm of Q values and a list of number of steps of episodes
    """
    # Initialize Q-values to zero
    q = np.zeros((mdp.nb_states, mdp.action_space.n)) # -------------- MAIN CHANGES FROM PREVIOUSLY

    # Initialize uniform policy
    policy = np.ones((mdp.nb_states, mdp.action_space.n)) / mdp.action_space.n

    # Store the number of steps and norms
    nb_steps = []
    value_norms = []

    for ep in range(nb_episodes):
        value_norm, steps = perform_episode(mdp, policy, q, alpha_critic, alpha_actor, update_type, render)
        value_norms.append(value_norm)
        nb_steps.append(steps)

    return value_norms, nb_steps

## Provide a plot function

Your plot function should show the evolution through time of number of steps the agent takes to find the reward in the maze.
If your algorithm works, this number of steps should decrease through time.

Your plot function should also show a mean and a standard deviation (or some more advanced statistics) over a collection of learning runs.
Make sure that your figure complies with [The figure checklist](https://master-dac.isir.upmc.fr/The_figure_checklist.pdf).

In [12]:
def plot_steps(
    update_type: int,
    nb_runs: int = 5,
    nb_episodes: int = 20,
    alpha_critic: float = 0.01,
    alpha_actor: float= 0.01,
    render: bool = True
)-> None:
    """
    Plot the performance of the Actor-Critic algorithm over multiple runs.
    Shows the mean and standard deviation of the number of steps per episode.
    :param update_type: int indicating the chosen update rule (1, 2, 3, 4)
    :param nb_episodes: the number of episodes
    :param alpha_critic: the learning rate of the critic
    :param alpha_actor: the learning rate of the actor
    :param render: whether training should be rendered
    :param nb_runs: number of runs to perform
    :return: nothing
    """
    all_steps = []
    all_vals = []

    #  faire tourner AC plusieurs (nb_runs) fois
    for run in range(nb_runs):
        vals, nb_steps = actor_critic_q(mdp, nb_episodes, alpha_critic, alpha_actor, update_type, render)
        all_steps.append(nb_steps)
        all_vals.append(vals)

    # récup les stats :
    all_steps = np.array(all_steps)
    mean_steps = np.mean(all_steps, axis=0)
    std_steps = np.std(all_steps, axis=0)

    all_vals = np.array(all_vals)
    mean_vals = np.mean(all_vals, axis=0)
    std_vals = np.std(all_vals, axis=0)

    # plot
    plt.figure()
    plt.title("Actor-Critic Performance - Mean Steps to Goal")
    plt.xlabel("Episode")
    plt.ylabel("Number of Steps")
    plt.grid(True, linestyle="--", alpha=0.6)

    # plot the mean and fill the standard deviation area
    plt.plot(mean_vals, label="Mean Vals", color="red")
    plt.plot(mean_steps, label="Mean Steps", color="blue")
    plt.fill_between(range(nb_episodes), mean_steps - std_steps, mean_steps + std_steps,
                     color="blue", alpha=0.2)

    plt.legend()
    plt.savefig("actor_critic_evol.pdf")

    plt.show()




## Actor-critic hyper-parameters

To represent the hyper-parameters of the experiments performed in this notebook, we suggest using the dictionary below.
This dictionary can be read using omegaconf.
Using it is not mandatory.
You can also change the value of hyper-parameters or environment parameters at will.

In [7]:
# default given initially
ac_params = {
    "save_curves": False,
    "save_heatmap": True,
    "mdp": {
        "name": "MazeMDP-v0",
        "width": 3,
        "height": 3,
        "ratio": 0.1,
        "render_mode": "rgb_array",
        },

    "log_dir": "./tmp",
    "video_dir": "./tmp/videos",

    "nb_episodes": 200,
    "render": False, # True, #
    "nb_repeats": 5,

    "alpha_critic": 0.5,
    "alpha_actor": 0.5,
    }

## Test your code

The code for testing what you did so far is provided below.

In [ ]:
def make_mdp(cfg):
    mdp = gym.make(cfg.mdp.name, kwargs={"width": cfg.mdp.width, "height": cfg.mdp.height, "ratio": cfg.mdp.ratio}, render_mode=cfg.mdp.render_mode)
    mdp = mdp.unwrapped
    return mdp

cfg=OmegaConf.create(ac_params)
mdp = make_mdp(cfg) # the mdp is made once and for all, this is an ugly global variable

import time
from datetime import timedelta

start_time = time.time()  # mesurer temps
# testing for first functions implemented :
# 1°: $Q(s_t, a_t) ← Q(s_t, a_t) + alpha-critic [r_{t+1} + \gamma Q(s_{t+1}, a_{t+1}) - Q(s_t, a_t)]$
plot_steps(1,nb_episodes=cfg.nb_episodes, nb_runs=cfg.nb_repeats, alpha_actor=cfg.alpha_actor, alpha_critic=cfg.alpha_critic)

# print("average v value:", v_vals.mean())

### More tests...

In [ ]:
ac_params = {
    "save_curves": False,
    "save_heatmap": True,
    "mdp": {
        "name": "MazeMDP-v0",
        "width": 5,
        "height": 5,
        "ratio": 0.2,
        "render_mode": "rgb_array",
        },

    "log_dir": "./tmp",
    "video_dir": "./tmp/videos",

    "nb_episodes": 100,
    "render": False, # True, #
    "nb_repeats": 5,

    "alpha_critic": 0.8, # ICI ON AUGMENTE LEARNING RATE DU CRITIC (because drives actor's LR)
    "alpha_actor": 0.5,
    }

cfg=OmegaConf.create(ac_params)
mdp = make_mdp(cfg) # the mdp is made once and for all, this is an ugly global variable

import time
from datetime import timedelta
N = 100

start_time = time.time()  # mesurer temps
plot_steps(1, nb_runs=N)
end_time = time.time()

elapsed_time = timedelta(seconds=(end_time - start_time))
print(f"Temps d'exécution pour {N} runs : {elapsed_time}")

# print("average v value:", v_vals.mean())

In [ ]:
ac_params = {
    "save_curves": False,
    "save_heatmap": True,
    "mdp": {
        "name": "MazeMDP-v0",
        "width": 5,
        "height": 5,
        "ratio": 0.2,
        "render_mode": "rgb_array",
        },

    "log_dir": "./tmp",
    "video_dir": "./tmp/videos",

    "nb_episodes": 100,
    "render": False, # True, #
    "nb_repeats": 5,

    "alpha_critic": 0.5, # CAS inverse du précédent (à titre d'observation)
    "alpha_actor": 0.8,
    }

cfg=OmegaConf.create(ac_params)
mdp = make_mdp(cfg) # the mdp is made once and for all, this is an ugly global variable

import time
from datetime import timedelta
N = 100

start_time = time.time()  # mesurer temps
plot_steps(1, nb_runs=N)
end_time = time.time()

elapsed_time = timedelta(seconds=(end_time - start_time))
print(f"Temps d'exécution pour {N} runs : {elapsed_time}")

# print("average v value:", v_vals.mean())

In [ ]:
gammas = [0.9, 0.95, 0.99]  # Valeurs de gamma à tester

for gamma in gammas:
    print(f"Test avec gamma = {gamma}")

    ac_params = {
        "save_curves": False,
        "save_heatmap": True,
        "mdp": {
            "name": "MazeMDP-v0",
            "width": 5,
            "height": 5,
            "ratio": 0.2,
            "render_mode": "rgb_array",
            "gamma": gamma,  # Ajouter gamma ici
        },
        "log_dir": "./tmp",
        "video_dir": "./tmp/videos",
        "nb_episodes": 100,
        "render": False,
        "nb_repeats": 5,
        "alpha_critic": 0.8,
        "alpha_actor": 0.5,
    }

    cfg = OmegaConf.create(ac_params)
    mdp = make_mdp(cfg)  # Création de l'environnement avec la nouvelle valeur de gamma

    import time
    from datetime import timedelta
    N = 100

    start_time = time.time()  # mesurer temps
    plot_steps(1, nb_runs=N)
    end_time = time.time()

    elapsed_time = timedelta(seconds=(end_time - start_time))
    print(f"Temps d'exécution pour {N} runs avec gamma = {gamma} : {elapsed_time}\n")


# Below, TO BE MODIFIED

# Step 2: Tuning hyper-parameters

In this part, you have to optimize two hyper-parameters of the actor-critic algorithm, namely the actor and critic learning rates.
You have to do so using a simple grid search method and some Bayesian optimization method.
For that, we suggest using [optuna](https://optuna.readthedocs.io/en/stable/).
Follow the above link to understand how optuna works.
The code to run optuna is provided.

You should make sure that the hyper-parameters tuning algorithms that you compare benefit from the same training budget
We suggest 400 training runs overall for each method,
which means 20 values each for the actor and the critic learning rates in the case of grid search.

By running the code below, you will do the following:

1. Perform hyper-parameters tuning with two algorithms as suggested above.

2. Provide a "heatmap" of the norm of the value function given the hyper-parameters, after training for each pair of hyper-parameters.

3. Collect the value of the best hyper-parameters found with each algorithm. You will need them for Step 3.

In [ ]:
def objective(trial):
    # Sample values of alpha_critic and alpha_actor
    alpha_critic = trial.suggest_float('alpha_critic', 0.01, 1.0)
    alpha_actor = trial.suggest_float('alpha_actor', 0.01, 1.0)

    # Run the actor-critic algorithm with sampled hyperparameters
    val_list, steps_list = actor_critic_v(
        mdp,
        alpha_critic=alpha_critic,
        alpha_actor=alpha_actor,
        nb_episodes=cfg.nb_episodes,
        render=False,  # Turn off rendering for faster runs
    )

    # We want to maximize the norm of the final value function ---> calcule de perf
    final_norm = val_list[-1]
    return final_norm

In [ ]:
# Bayesian optimization
study_Bayes = optuna.create_study(direction='maximize')
study_Bayes.optimize(objective, n_trials=400)

# Data frame contains the params and the corresponding norm of the final value function
study_Bayes_analyse = study_Bayes.trials_dataframe(attrs=('params', 'value'))

best_Bayes = study_Bayes.best_params
print ('The best parameters founded using Bayesian optimization are: ', best_Bayes, '\n\n')

# Heat map
plt.figure(figsize=(8, 4))
plt.scatter(study_Bayes_analyse['params_alpha_critic'],
             study_Bayes_analyse['params_alpha_actor'],
             c=study_Bayes_analyse['value'], cmap="RdYlGn_r")
plt.title('Value function norms (Bayesian optimization)')
plt.xlabel(r'$\alpha_{critic}$')
plt.ylabel(r'$\alpha_{actor}$')
plt.colorbar()
plt.savefig("bayes_opt.pdf")
# plt.show()

In [ ]:
# Grid search
search_space = {'alpha_critic': np.linspace(0.01, 1.0, 20), 'alpha_actor': np.linspace(0.01, 1.0, 20)}
study_grid = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study_grid.optimize(objective)

# Data frame contains the params and the corresponding norm of the final value function
study_grid_analyse = study_grid.trials_dataframe(attrs=('params', 'value'))

best_grid = study_grid.best_params
print ('The best parameters founded using Grid search are: ', best_grid, '\n\n')

# Heat map
plt.figure(figsize=(8, 4))
plt.scatter(study_grid_analyse['params_alpha_critic'],
             study_grid_analyse['params_alpha_actor'],
             c=study_grid_analyse['value'], cmap="RdYlGn_r")
plt.title('Value function norms (Grid search)')
plt.xlabel(r'$\alpha_{critic}$')
plt.ylabel(r'$\alpha_{actor}$')
plt.grid(False)
plt.colorbar()
plt.savefig("grid_search.pdf")
# plt.show()

# Step 3: Statistical tests

Now you have to compare the performance of the actor-critic algorithm tuned
with all the best hyper-parameters you found before, using statistical tests.

The functions below are provided to run Welch's T-test over learning curves.
They have been adapted from a github repository: https://github.com/flowersteam/rl_stats
You don't need to understand them in detail (though it is always a good idea to try to understand more code).

In [ ]:
from scipy.stats import ttest_ind
import bootstrapped.bootstrap as bs
import bootstrapped.compare_functions as bs_compare
import bootstrapped.stats_functions as bs_stats

In [ ]:
def compute_central_tendency_and_error(id_central, id_error, sample):

    try:
        id_error = int(id_error)
    except:
        pass

    if id_central == "mean":
        central = np.nanmean(sample, axis=1)
    elif id_central == "median":
        central = np.nanmedian(sample, axis=1)
    else:
        raise NotImplementedError

    if isinstance(id_error, int):
        low = np.nanpercentile(sample, q=int((100 - id_error) / 2), axis=1)
        high = np.nanpercentile(sample, q=int(100 - (100 - id_error) / 2), axis=1)
    elif id_error == "std":
        low = central - np.nanstd(sample, axis=1)
        high = central + np.nanstd(sample, axis=1)
    elif id_error == "sem":
        low = central - np.nanstd(sample, axis=1) / np.sqrt(sample.shape[0])
        high = central + np.nanstd(sample, axis=1) / np.sqrt(sample.shape[0])
    else:
        raise NotImplementedError

    return central, low, high

def run_test(data1, data2, alpha=0.05):
    """
    Compute tests comparing data1 and data2 with confidence level alpha
    :param data1: (np.ndarray) sample 1
    :param data2: (np.ndarray) sample 2
    :param alpha: (float) confidence level of the test
    :return: (bool) if True, the null hypothesis is rejected
    """
    data1 = data1.squeeze()
    data2 = data2.squeeze()

    # perform Welch t-test":
    _, p = ttest_ind(data1, data2, equal_var=False)
    return p < alpha

This last function was adapted for the lab.

In [ ]:
def perform_test(perf1, perf2, name1, name2, sample_size=20, downsampling_fact=5, confidence_level=0.01):

    perf1 = perf1.transpose()
    perf2 = perf2.transpose()
    nb_datapoints = perf1.shape[1]
    nb_steps = perf1.shape[0]

    legend = [name1, name2]

    # what do you want to plot ?
    id_central = 'mean' # "median"  #
    id_error = 80  # (percentiles), also: 'std', 'sem'

    sample1 = perf1[:, np.random.randint(0, nb_datapoints, sample_size)]
    sample2 = perf2[:, np.random.randint(0, nb_datapoints, sample_size)]

    steps = np.arange(0, nb_steps, downsampling_fact)
    sample1 = sample1[steps, :]
    sample2 = sample2[steps, :]

    # test
    sign_diff = np.zeros([len(steps)])
    for i in range(len(steps)):
        sign_diff[i] = run_test(
            sample1[i, :], sample2[i, :], alpha=confidence_level
        )

    central1, low1, high1 = compute_central_tendency_and_error(
        id_central, id_error, sample1
    )
    central2, low2, high2 = compute_central_tendency_and_error(
        id_central, id_error, sample2
    )

    # plot
    _, ax = plt.subplots(1, 1, figsize=(20, 10))
    lab1 = plt.xlabel("training steps")
    lab2 = plt.ylabel("performance")

    plt.plot(steps*downsampling_fact, central1, linewidth=10)
    plt.plot(steps*downsampling_fact, central2, linewidth=10)
    plt.fill_between(steps*downsampling_fact, low1, high1, alpha=0.3)
    plt.fill_between(steps*downsampling_fact, low2, high2, alpha=0.3)
    leg = ax.legend(legend, frameon=False)

    # plot significative difference as dots
    idx = np.argwhere(sign_diff == 1)
    y = max(np.nanmax(high1), np.nanmax(high2))
    # Plot the stars where there is a statistically significant difference
    plt.scatter(steps[idx]*downsampling_fact, y * 1.05 * np.ones([idx.size]), s=100, c="k", marker="*")

    # style
    for line in leg.get_lines():
        line.set_linewidth(10.0)
    ax.spines["top"].set_linewidth(5)
    ax.spines["right"].set_linewidth(5)
    ax.spines["bottom"].set_linewidth(5)
    ax.spines["left"].set_linewidth(5)

    plt.savefig(
        f"./{name1}_{name2}.png", bbox_extra_artists=(leg, lab1, lab2), bbox_inches="tight", dpi=100
    )
    # plt.show()

It is now time to compare the performance of your actor-critic algorithm, using different hyper-parameters.
As hyper-parameters, you will use:

- naive tuning, that is a pair (0.5, 0.5) for the actor and critic learning rates,
- the best hyper-parameters you found with the different tuning algorithms you used before.

Below, we provide the code for comparing the Bayesian optimization approach and grid search.
You have to extend it to also compare these two approaches to using naive tuning.

The code below does the following:

1. For each set of hyper-parameters, collect a large dataset of learning curves (we suggest using 150 training episodes)

2. Perform statistical comparisons

- Take two datasets of learning curves obtained with the hyper-parameters sets that you found with different tuning algorithms.
- Use the ``` perform_test(...)``` function to compare each possible pair of sets.

You should obtain an image for each pair you have tried.
In this image, black dots signal the time step where there is a statistically significant difference between two learning curves.

In [ ]:
nb_runs = 150
# Collect dataset
perf_naive = [] # Dataset for naive tuning
perf_Bayes = [] # Dataset for Bayesian optimization
perf_grid = [] # Dataset for Grid search

for _ in range(nb_runs):
    #----------------------------------------------------------------------------------
    v_naive, _ = actor_critic_v(mdp, nb_episodes=cfg.nb_episodes,
                                alpha_critic=0.5, alpha_actor=0.5, render=False)
    #----------------------------------------------------------------------------------
    v_Bayes, _ = actor_critic_v(mdp, nb_episodes=cfg.nb_episodes,
                                   alpha_critic=best_Bayes['alpha_critic'], alpha_actor=best_Bayes['alpha_actor'],
                                   render=False)
    v_grid, _ = actor_critic_v(mdp, nb_episodes=cfg.nb_episodes,
                                   alpha_critic=best_grid['alpha_critic'], alpha_actor=best_grid['alpha_actor'],
                                   render=False)
    perf_naive.append(v_naive) #this
    perf_Bayes.append(v_Bayes)
    perf_grid.append(v_grid)

perf_naive = np.array(perf_naive) #this
perf_Bayes = np.array(perf_Bayes)
perf_grid = np.array(perf_grid)

# Perform statistical comparisons
perform_test(perf_Bayes, perf_grid, 'Bayes optimization', 'Grid search')
perform_test(perf_naive, perf_Bayes, 'Bayes optimization', 'Naive tuning') #this
perform_test(perf_naive, perf_grid, 'Naive tuning', 'Grid search') #this

# To be completed...
# assert False, 'Not implemented yet'
# Once you have completed the code, conclude